# Arabica Coffee C Futures Return Model Workflow

This notebook documents the complete workflow used in this project:

- inspect `data/centralData/yahoo_cot_full_outer_by_date.csv`
- refresh Open-Meteo weather data for coffee-growing regions
- build leak-aware price, COT, weather, and calendar features
- train and save the 5-trading-day return model
- evaluate accuracy, ROC, real-vs-predicted price, feature importance, drift, and COT correlations
- add a separate dated news context layer for the model's worst error dates
- load the saved model for latest predictions

The notebook reuses the project scripts and modules so the logic stays in one maintained place.

## 1. Setup

Run from the repository root. If using Jupyter, select the `.venv` kernel if available.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "data").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.append(str(ROOT / "Scripts" / "project" / "src"))

print("Project root:", ROOT)
print("Python:", sys.executable)

## 2. Inspect the central data file

In [ ]:
central_path = ROOT / "data" / "centralData" / "yahoo_cot_full_outer_by_date.csv"
df = pd.read_csv(central_path, low_memory=False)

summary = {
    "rows": len(df),
    "columns": len(df.columns),
    "date_min": df["Date"].min(),
    "date_max": df["Date"].max(),
    "rows_with_close": int(df["Close"].notna().sum()),
    "rows_with_cot": int(df["cot_report_date"].notna().sum()),
    "rows_with_future_return_5d": int(df["future_return_5d"].notna().sum()),
}
summary

In [ ]:
display(df[[
    "Date", "date_match_status", "cot_report_date", "Open", "High", "Low", "Close", "Volume",
    "return_1d", "return_5d", "future_return_5d", "future_direction_5d",
    "managed_money_net", "commercial_net", "noncommercial_net", "open_interest_change_pct"
]].head(10))

display(df["date_match_status"].value_counts(dropna=False).to_frame("rows"))

## 3. Refresh weather data

This fetches Open-Meteo archive data for:

- Brazil Minas Gerais
- Colombia Huila
- Vietnam Dak Lak

The refresh script preserves an existing good cache and writes per-region files. It may need network access if run inside a restricted environment.

In [ ]:
# Run this cell when you want to refresh missing weather regions.
# It is safe to rerun; it detects already-present regions unless --force-all is passed.
cmd = [sys.executable, "Scripts/project/scripts/refresh_weather_cache.py"]
print("Command:", " ".join(cmd))
# subprocess.run(cmd, check=True)

In [ ]:
weather_path = ROOT / "data" / "weather" / "open_meteo_coffee_regions_daily.csv"
weather = pd.read_csv(weather_path)
regions = ["brazil_minas_gerais", "colombia_huila", "vietnam_dak_lak"]

print("Weather shape:", weather.shape)
print("Weather date range:", weather["Date"].min(), "to", weather["Date"].max())
for region in regions:
    cols = [c for c in weather.columns if c.startswith(f"weather_{region}_")]
    print(region, "columns:", len(cols), "non-null rows:", int(weather[cols[0]].notna().sum()) if cols else 0)

display(weather.head())

## 4. Build modeling frame

Important modeling choices:

- target is future 5-trading-day return
- COT reports are lagged by 3 calendar days before being available to the model
- source/archive/code metadata and raw OHLCV price levels are excluded
- current close is still retained outside the feature matrix to convert predicted returns into predicted future prices

In [ ]:
from arabica_modeling import (
    TARGET_CLOSE,
    TARGET_DIRECTION,
    TARGET_RETURN,
    build_modeling_frame,
    chronological_split,
    feature_frame_for_model,
)

feature_set = build_modeling_frame(central_path, weather_path=weather_path)
model_frame = feature_set.frame.replace([np.inf, -np.inf], np.nan)
train, valid, test = chronological_split(model_frame)

print("Model rows:", len(model_frame))
print("Feature count:", len(feature_set.numeric_features))
print("Dropped feature count:", len(feature_set.dropped_features))
print("Train range:", train["Date"].min(), "to", train["Date"].max())
print("Valid range:", valid["Date"].min(), "to", valid["Date"].max())
print("Test range:", test["Date"].min(), "to", test["Date"].max())

display(pd.Series(feature_set.dropped_features, name="dropped_features").to_frame().head(40))

## 5. Train and save the model

This is the exact training entry point used for the current model artifacts. It trains candidate regressors/classifiers, saves the best model bundle, metrics JSON, holdout predictions, and all plots.

In [ ]:
train_cmd = [sys.executable, "Scripts/project/scripts/train_arabica_model.py"]
print("Command:", " ".join(train_cmd))
# subprocess.run(train_cmd, check=True)

## 6. Load saved metrics

In [ ]:
metrics_path = ROOT / "Scripts" / "project" / "artifacts" / "outputs" / "arabica_model_metrics.json"
metrics = json.loads(metrics_path.read_text())

print("Best regressor:", metrics["model_selection"]["best_regressor"])
print("Best classifier:", metrics["model_selection"]["best_classifier"])
print("Holdout regression:")
display(pd.Series(metrics["holdout_test"]["regression"]).to_frame("value"))
print("Holdout classification:")
display(pd.Series(metrics["holdout_test"]["classification"]).to_frame("value"))
print("Feature metadata:")
display(pd.Series({k: v for k, v in metrics["features"].items() if k != "dropped_features"}).to_frame("value"))

## 7. Display core evaluation plots

In [ ]:
plot_files = [
    "regression_metrics.png",
    "classification_accuracy_metrics.png",
    "roc_curve.png",
    "actual_vs_predicted_returns.png",
    "real_vs_predicted_price.png",
    "feature_importance.png",
]

for plot_file in plot_files:
    path = ROOT / "Scripts" / "project" / "artifacts" / "plots" / plot_file
    if path.exists():
        display(Markdown(f"### {plot_file}"))
        display(Image(filename=str(path)))

## 8. Drift and COT diagnostics

These outputs answer: where did prediction error drift occur, which features changed most from train/validation to holdout, and whether COT features correlated with target returns or forecast errors.

In [ ]:
drift_report = pd.read_csv(ROOT / "Scripts" / "project" / "artifacts" / "outputs" / "feature_drift_cot_correlation_report.csv")
worst_dates = pd.read_csv(ROOT / "Scripts" / "project" / "artifacts" / "outputs" / "worst_error_dates_with_drift_context.csv")

display(drift_report.head(25))
display(worst_dates.head(25))

In [ ]:
diagnostic_plots = [
    "prediction_error_drift.png",
    "rolling_direction_accuracy.png",
    "feature_drift_top.png",
    "cot_target_correlation.png",
    "cot_error_correlation.png",
    "cot_drift_vs_error_correlation.png",
]

for plot_file in diagnostic_plots:
    path = ROOT / "Scripts" / "project" / "artifacts" / "plots" / plot_file
    if path.exists():
        display(Markdown(f"### {plot_file}"))
        display(Image(filename=str(path)))

## 9. News context overlay

This layer is intentionally separate from the base model. It searches dated coffee news around the worst model-error dates and scores whether the news was bullish/bearish enough to plausibly affect weekly returns.

The default is deterministic keyword scoring. Ollama is optional and only used when `--use-ollama` is passed.

In [ ]:
news_cmd = [sys.executable, "Scripts/project/scripts/news_context_analysis.py", "--top-n", "25", "--max-articles", "8"]
print("Command:", " ".join(news_cmd))
# subprocess.run(news_cmd, check=True)

# Optional Ollama-assisted version. Requires data/centralData/.env with OLLAMA_API_KEY if your Ollama endpoint needs it.
# ollama_cmd = news_cmd + ["--use-ollama", "--ollama-model", "llama3.1"]
# subprocess.run(ollama_cmd, check=True)

In [ ]:
news_summary = pd.read_csv(ROOT / "Scripts" / "project" / "artifacts" / "outputs" / "news_context_by_error_date.csv")
news_articles = pd.read_csv(ROOT / "Scripts" / "project" / "artifacts" / "outputs" / "news_articles_by_error_date.csv")

display(news_summary[[
    "Date", "actual_return_5d", "abs_return_error", "article_count", "weighted_news_score",
    "max_weekly_strength_score", "news_direction", "news_aligned_with_actual_move",
    "strong_weekly_news_effect", "top_news_titles"
]].head(25))

display(news_articles.head(25))

In [ ]:
news_plots = [
    "news_impact_vs_return.png",
    "news_strength_vs_model_error.png",
    "news_article_count_by_error_date.png",
]

for plot_file in news_plots:
    path = ROOT / "Scripts" / "project" / "artifacts" / "plots" / plot_file
    if path.exists():
        display(Markdown(f"### {plot_file}"))
        display(Image(filename=str(path)))

## 10. Use the saved model for prediction

In [ ]:
predict_cmd = [sys.executable, "Scripts/project/scripts/predict_arabica_returns.py", "--latest-rows", "10"]
print("Command:", " ".join(predict_cmd))
# subprocess.run(predict_cmd, check=True)

latest = pd.read_csv(ROOT / "Scripts" / "project" / "artifacts" / "outputs" / "latest_arabica_predictions.csv")
display(latest)

## 11. Inspect exact project source code

Run this cell to print the implementation files used by the notebook. This keeps the notebook compact while still making the full code auditable from the notebook.

In [ ]:
source_files = [
    "Scripts/project/src/arabica_modeling.py",
    "Scripts/project/scripts/refresh_weather_cache.py",
    "Scripts/project/scripts/train_arabica_model.py",
    "Scripts/project/scripts/predict_arabica_returns.py",
    "Scripts/project/scripts/news_context_analysis.py",
]

for file_name in source_files:
    path = ROOT / file_name
    display(Markdown(f"### `{file_name}`"))
    print(path.read_text())
    print("\n" + "=" * 100 + "\n")